In [1]:
import os
import cv2
from pathlib import Path
from tqdm.auto import tqdm

# --- CONFIGURATION ---
MERGED_DIR = Path("merged_dataset")
VIT_DIR = Path("vit_dataset")

# Unified class names matching your UNIFIED_YAML
CLASS_NAMES = {
    0: "cattle",
    1: "lumpy_skin",
    2: "foot_and_mouth",
    3: "mastitis"
}

def create_vit_dataset():
    """
    Parses YOLO bounding box annotations, crops regions of interest (ROIs),
    and organizes them into class subfolders for ViT training.
    """
    total_crops_generated = 0

    for split in ['train', 'val', 'test']:
        img_dir = MERGED_DIR / split / 'images'
        lbl_dir = MERGED_DIR / split / 'labels'

        if not img_dir.exists():
            print(f"⚠️ Skipping '{split}' split (directory not found).")
            continue

        # Create output class directories for ViT
        for class_name in CLASS_NAMES.values():
            (VIT_DIR / split / class_name).mkdir(parents=True, exist_ok=True)

        image_files = list(img_dir.glob('*'))
        print(f"\n✂️ Cropping {split} split ({len(image_files)} images)...")

        for img_path in tqdm(image_files, desc=f"Processing {split}"):
            # Match corresponding .txt annotation file
            lbl_path = lbl_dir / f"{img_path.stem}.txt"

            if not lbl_path.exists():
                continue

            # Read image
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            img_h, img_w = img.shape[:2]

            # Read bounding box lines
            with open(lbl_path, 'r') as f:
                lines = f.readlines()

            for crop_idx, line in enumerate(lines):
                parts = line.strip().split()
                if len(parts) < 5:
                    continue  # Skip empty background files or invalid lines

                class_id = int(parts[0])
                if class_id not in CLASS_NAMES:
                    continue

                class_name = CLASS_NAMES[class_id]

                # De-normalize YOLO coordinates (0.0 - 1.0) to absolute pixel bounds
                x_center, y_center, w, h = map(float, parts[1:5])

                x1 = int(max(0, (x_center - w / 2) * img_w))
                y1 = int(max(0, (y_center - h / 2) * img_h))
                x2 = int(min(img_w, (x_center + w / 2) * img_w))
                y2 = int(min(img_h, (y_center + h / 2) * img_h))

                # Extract cropped ROI
                crop = img[y1:y2, x1:x2]

                # Ensure the crop is valid (non-zero width/height)
                if crop.size > 0:
                    crop_filename = VIT_DIR / split / class_name / f"{img_path.stem}_crop{crop_idx}.jpg"
                    cv2.imwrite(str(crop_filename), crop)
                    total_crops_generated += 1

    print(f"\n✅ ViT Dataset Creation Complete!")
    print(f"Total cropped ROI samples generated: {total_crops_generated}")
    print(f"Dataset saved to: {VIT_DIR.resolve()}")

if __name__ == '__main__':
    create_vit_dataset()


✂️ Cropping train split (6958 images)...


Processing train:   0%|          | 0/6958 [00:00<?, ?it/s]


✂️ Cropping val split (1130 images)...


Processing val:   0%|          | 0/1130 [00:00<?, ?it/s]


✂️ Cropping test split (942 images)...


Processing test:   0%|          | 0/942 [00:00<?, ?it/s]


✅ ViT Dataset Creation Complete!
Total cropped ROI samples generated: 11316
Dataset saved to: C:\Users\lasit\vit_dataset


In [26]:
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from PIL import ImageOps
from tqdm.auto import tqdm

# --- CONFIGURATION ---
DATASET_DIR = r"C:\Users\lasit\vit_dataset"
MODEL_SAVE_PATH = "best_vit_model.pth"
BATCH_SIZE = 32           # Safe for 4GB VRAM with FP16
EPOCHS = 50               # Maximum epochs; early stopping will cut off when converged
PATIENCE = 10             # Early stopping patience
LEARNING_RATE = 1e-4      # Standard fine-tuning learning rate for transformers
WEIGHT_DECAY = 0.01       # AdamW regularization
NUM_WORKERS = 0           # 0 prevents PyTorch multiprocessing memory crashes on Windows

# --- ASPECT RATIO PRESERVING PAD ---
class SquarePad:
    """Pads rectangular ROI crops to a square before resizing to prevent texture stretching."""
    def __call__(self, image):
        w, h = image.size
        max_wh = max(w, h)
        hp = int((max_wh - w) / 2)
        vp = int((max_wh - h) / 2)
        padding = (hp, vp, hp, vp)
        return ImageOps.expand(image, padding, fill=(114, 114, 114))

# --- EARLY STOPPING CLASS ---
class EarlyStopping:
    def __init__(self, patience=10, path="best_vit_model.pth"):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            print(f"  📈 Validation loss improved ({self.best_loss:.4f} --> {val_loss:.4f}). Saving model...")
            self.best_loss = val_loss
            torch.save(model.state_dict(), self.path)
            self.counter = 0
        else:
            self.counter += 1
            print(f"  ⚠️ EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

# --- MAIN TRAINING PIPELINE ---
def main():
    # 1. Hardware Verification
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Using Device: {device}")
    if device.type == 'cuda':
        print(f"GPU Model: {torch.cuda.get_device_name(0)}")

    # 2. Data Transforms (SquarePad + 224x224 Resize + ImageNet Normalization)
    train_transforms = transforms.Compose([
        SquarePad(),                      # Preserve natural aspect ratio
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_test_transforms = transforms.Compose([
        SquarePad(),                      # Preserve natural aspect ratio
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 3. Load Datasets
    print("\n📂 Loading datasets...")
    train_dataset = datasets.ImageFolder(os.path.join(DATASET_DIR, 'train'), transform=train_transforms)
    val_dataset = datasets.ImageFolder(os.path.join(DATASET_DIR, 'val'), transform=val_test_transforms)
    test_dataset = datasets.ImageFolder(os.path.join(DATASET_DIR, 'test'), transform=val_test_transforms)

    class_names = train_dataset.classes
    num_classes = len(class_names)
    print(f"Classes Found ({num_classes}): {class_names}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # 4. Load Pretrained ViT & Modify Head
    print("\n🧠 Initializing Pretrained Vision Transformer (ViT-B/16)...")
    model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)

    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    model = model.to(device)

    # 5. Class-Weighted Loss Function & Optimizer
    # Alphabetical Class Order: ['cattle', 'foot_and_mouth', 'lumpy_skin', 'mastitis']
    # Inverse class weights calculated from training set distribution:
    class_weights = torch.tensor([0.40, 2.87, 1.05, 5.32]).to(device)
    print(f"⚖️ Applied Inverse Class Loss Weights: {class_weights.tolist()}")

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler()  # Automatic Mixed Precision
    early_stopping = EarlyStopping(patience=PATIENCE, path=MODEL_SAVE_PATH)

    # 6. Training Loop
    print("\n🔥 Starting Fine-Tuning...")
    start_time = time.time()

    for epoch in range(EPOCHS):
        # --- TRAIN PHASE ---
        model.train()
        train_loss, train_correct = 0.0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == labels.data)

        scheduler.step()
        epoch_train_loss = train_loss / len(train_dataset)
        epoch_train_acc = train_correct.double() / len(train_dataset)

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss, val_correct = 0.0, 0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
                images, labels = images.to(device), labels.to(device)

                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data)

        epoch_val_loss = val_loss / len(val_dataset)
        epoch_val_acc = val_correct.double() / len(val_dataset)

        print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} | "
              f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        # Check Early Stopping
        early_stopping(epoch_val_loss, model)
        if early_stopping.early_stop:
            print("\n🛑 Early stopping triggered. Training halted.")
            break

    elapsed_time = time.time() - start_time
    print(f"\n✨ Training Complete in {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s!")

    # 7. Final Test Evaluation
    print("\n📊 Evaluating Best Checkpoint on Test Set...")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval()

    test_correct = 0
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Testing"):
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
            _, preds = torch.max(outputs, 1)
            test_correct += torch.sum(preds == labels.data)

    test_acc = test_correct.double() / len(test_dataset)
    print("\n==========================================")
    print(f"🏆 Final Test Set Accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")
    print(f"💾 Best Weights Saved To: {os.path.abspath(MODEL_SAVE_PATH)}")
    print("==========================================")

if __name__ == '__main__':
    main()

🚀 Using Device: cuda:0
GPU Model: NVIDIA GeForce RTX 3050 Laptop GPU

📂 Loading datasets...
Classes Found (4): ['cattle', 'foot_and_mouth', 'lumpy_skin', 'mastitis']

🧠 Initializing Pretrained Vision Transformer (ViT-B/16)...
⚖️ Applied Inverse Class Loss Weights: [0.4000000059604645, 2.869999885559082, 1.0499999523162842, 5.320000171661377]

🔥 Starting Fine-Tuning...


C:\Users\lasit\AppData\Local\Temp\ipykernel_26920\503381067.py:110: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Automatic Mixed Precision


Epoch 1/50 [Train]:   0%|          | 0/266 [00:00<?, ?it/s]

C:\Users\lasit\AppData\Local\Temp\ipykernel_26920\503381067.py:126: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/50 [Val]:   0%|          | 0/49 [00:00<?, ?it/s]

...

Epoch 13/50 [Val]:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 13/50 | Train Loss: 0.0569 Acc: 0.9665 | Val Loss: 0.5096 Acc: 0.8613
  ⚠️ EarlyStopping counter: 10/10

🛑 Early stopping triggered. Training halted.

✨ Training Complete in 203m 52s!

📊 Evaluating Best Checkpoint on Test Set...


Testing:   0%|          | 0/40 [00:00<?, ?it/s]

C:\Users\lasit\AppData\Local\Temp\ipykernel_26920\503381067.py:183: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



🏆 Final Test Set Accuracy: 0.9011 (90.11%)
💾 Best Weights Saved To: C:\Users\lasit\best_vit_model.pth


In [38]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import vit_b_16
from PIL import Image
import tkinter as tk
from tkinter import filedialog

# --- CONFIGURATION ---
MODEL_PATH = "best_vit_model.pth"

# PyTorch's ImageFolder automatically sorts classes alphabetically
CLASS_NAMES = ['cattle', 'foot_and_mouth', 'lumpy_skin', 'mastitis']

def load_vit_model(model_path: str, num_classes: int = 4):
    """Loads the fine-tuned ViT model architecture and trained weights."""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Loading ViT Model on: {device}")

    # 1. Initialize ViT-B/16 architecture
    model = vit_b_16(weights=None)

    # 2. Re-create the custom 4-class classification head
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)

    # 3. Load trained weights
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"❌ Checkpoint file not found at: {os.path.abspath(model_path)}")

    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model, device

def predict_image(image_path: str, model, device):
    """Preprocesses the input image and outputs classification probabilities."""
    # ViT preprocessing: 224x224 resize & ImageNet normalization
    test_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Load raw image
    raw_img = Image.open(image_path).convert('RGB')
    input_tensor = test_transforms(raw_img).unsqueeze(0).to(device)

    # Run inference
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    # Extract top prediction
    top_prob, top_class_idx = torch.topk(probabilities, 1)
    predicted_class = CLASS_NAMES[top_class_idx.item()]
    confidence = top_prob.item() * 100

    # Print results summary
    print("\n==================================================")
    print(f"📸 Tested Image: {os.path.basename(image_path)}")
    print(f"🎯 Predicted Class: {predicted_class.upper()}")
    print(f"📊 Confidence Score: {confidence:.2f}%")
    print("==================================================")
    print("Full Probability Breakdown:")
    for idx, prob in enumerate(probabilities):
        print(f"  • {CLASS_NAMES[idx]:<16}: {prob.item() * 100:6.2f}%")
    print("==================================================")

def run_test():
    # Hide the main tkinter root window and force popup to top
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)

    # Open File Dialog Box
    file_path = filedialog.askopenfilename(
        title="Select Image Crop for Vision Transformer Diagnosis",
        filetypes=[("Image files", "*.jpg *.jpeg *.png *.webp *.bmp")]
    )

    if not file_path:
        print("❌ File selection canceled.")
        return

    model, device = load_vit_model(MODEL_PATH, num_classes=len(CLASS_NAMES))
    predict_image(file_path, model, device)

if __name__ == "__main__":
    run_test()

🚀 Loading ViT Model on: cuda:0

📸 Tested Image: images.jpg
🎯 Predicted Class: MASTITIS
📊 Confidence Score: 91.58%
Full Probability Breakdown:
  • cattle          :   8.09%
  • foot_and_mouth  :   0.02%
  • lumpy_skin      :   0.31%
  • mastitis        :  91.58%
